# Input summary statistics

This notebook generates the input summary statistics in gentropy format from SuShiE test data files.

The data is saved into the `/testdata` directory

In [8]:
from pyspark.sql import types as t
from pyspark.sql import functions as f
from pyspark.sql import Column, DataFrame
import random
import string
from pyspark.sql import SparkSession
from typing import NamedTuple
import sys
import pandas as pd

random.seed(42)


In [2]:
def random_string(length: int = 10) -> str:
    chars = string.ascii_letters + string.digits
    return "".join(random.choice(chars) for _ in range(length))


class PValComponents(NamedTuple):
    mantissa: Column
    exponent: Column


def split_pvalue(pv: Column) -> PValComponents:
    """Split pvalue for mantissa and exponent"""
    pv = f.when(
        pv == f.lit("0"), f.lit(sys.float_info.min).cast(t.StringType())
    ).otherwise(pv)

    # Get exponent:
    exponent = f.when(
        f.upper(pv).contains("E"),
        f.split(f.upper(pv), "E").getItem(1),
    ).otherwise(f.floor(f.log10(pv)))

    # Get mantissa:
    mantissa = f.when(
        f.upper(pv).contains("E"),
        f.split(f.upper(pv), "E").getItem(0),
    ).otherwise(pv / (10**exponent))

    # Round value:
    mantissa = f.round(mantissa, 3)

    return PValComponents(
        mantissa=mantissa.cast(t.FloatType()).alias("pValueMantissa"),
        exponent=exponent.cast(t.IntegerType()).alias("pValueExponent"),
    )


def transform_to_gentropy_ss(df: DataFrame) -> DataFrame:
    """Transform gwas to gentropy summary statistics"""
    return (
        df.withColumn("studyId", f.lit(random_string(10)))
        .withColumn("chromosome", f.col("chrom").cast(t.StringType()))
        .withColumn(
            "variantId",
            f.concat_ws(
                "_", f.col("chromosome"), f.col("pos"), f.col("a1"), f.col("a0")
            ),
        )
        .withColumn("position", f.col("pos").cast(t.IntegerType()))
        .withColumn(
            "beta", f.col("z").cast(t.DoubleType()) * f.col("se").cast(t.DoubleType())
        )
        .withColumn("sampleSize", f.lit(None).cast(t.IntegerType()))
        .withColumn(
            "pValueMantissa", split_pvalue(f.col("pval").cast(t.DoubleType())).mantissa
        )
        .withColumn(
            "pValueExponent", split_pvalue(f.col("pval").cast(t.DoubleType())).exponent
        )
        .withColumn("effectAlleleFrequencyFromSource", f.lit(None).cast(t.FloatType()))
        .withColumn("standardError", f.col("se").cast(t.DoubleType()))
        .select(
            "studyId",
            "variantId",
            "chromosome",
            "position",
            "beta",
            "sampleSize",
            "pValueMantissa",
            "pValueExponent",
            "effectAlleleFrequencyFromSource",
            "standardError",
        )
        .cache()
    )


In [3]:
session = SparkSession.builder.getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 11:07:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/12 11:07:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
eur_gwas = (
    "https://raw.githubusercontent.com/mancusolab/sushie/refs/heads/main/data/EUR.gwas"
)
afr_gwas = (
    "https://raw.githubusercontent.com/mancusolab/sushie/refs/heads/main/data/AFR.gwas"
)
eur_ld = (
    "https://raw.githubusercontent.com/mancusolab/sushie/refs/heads/main/data/EUR.ld"
)
afr_ld = (
    "https://raw.githubusercontent.com/mancusolab/sushie/refs/heads/main/data/AFR.ld"
)


In [5]:
schema = t.StructType(
    [
        t.StructField("studyId", t.StringType(), False),
        t.StructField("variantId", t.StringType(), False),
        t.StructField("chromosome", t.StringType(), False),
        t.StructField("position", t.IntegerType(), False),
        t.StructField("beta", t.DoubleType(), False),
        t.StructField("sampleSize", t.IntegerType(), True),
        t.StructField("pValueMantissa", t.FloatType(), False),
        t.StructField("pValueExponent", t.IntegerType(), False),
        t.StructField("effectAlleleFrequencyFromSource", t.FloatType(), True),
        t.StructField("standardError", t.DoubleType(), True),
    ]
)


In [6]:
eur_gwas_df = session.createDataFrame(pd.read_csv(eur_gwas, sep="\t")).transform(
    transform_to_gentropy_ss
)
eur_gwas_df.coalesce(1).write.mode("overwrite").partitionBy("studyId").parquet(
    "../../testdata/eur_gwas"
)


In [7]:
afr_gwas_df = session.createDataFrame(pd.read_csv(afr_gwas, sep="\t")).transform(
    transform_to_gentropy_ss
)
afr_gwas_df.coalesce(1).write.mode("overwrite").partitionBy("studyId").parquet(
    "../../testdata/afr_gwas"
)
